In [ ]:
from IPython.display import clear_output

%pip install kagglehub catboost tqdm -q

clear_output()

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from tqdm import tqdm
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error as mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor

%matplotlib inline

In [ ]:
# Task 1: Write your code here:
csv_path = os.path.join(path, "Q1_data.csv")

df = pd.read_csv(csv_path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
df["Delivery_Time"].hist(bins=40, edgecolor='black')

plt.title(f"Target Distribution - Delivery Time")
plt.xlabel("Delivery Time")
plt.ylabel("Frequency")
plt.grid(False)

plt.show()

In [ ]:
df.head() #For checking dropped column

In [ ]:
# Task 1: Write your code here:
df.drop(columns=["Order_ID"], inplace=True)
df.head()

In [ ]:
df.info()

In [ ]:
# Task 2: Write your code here:
# Handle missing values
df['Weather'].fillna(df['Weather'].mode()[0], inplace=True)
df['Traffic_Level'].fillna(df['Traffic_Level'].mode()[0], inplace=True)
df['Time_of_Day'].fillna(df['Time_of_Day'].mode()[0], inplace=True)

df['Courier_Experience_yrs'].fillna(df['Courier_Experience_yrs'].median(), inplace=True)

df.dropna(subset=['Delivery_Time'], inplace=True) # If target isnt available drop row

In [ ]:
df.info()

In [ ]:
# Task 3: Write your code here:
duplicates = df.duplicated().sum()
print(f"Number of Duplicate Samples: {duplicates}")
if duplicates > 0:
  print("Dropping Duplicates...")
  df.drop_duplicates(inplace=True)
  print("Duplicates Dropped.")
else:
  print("No Duplicate Samples Found.")

In [ ]:
# Task 4: Write your code here:
# Enoding OHE
ohe_cols = ["Weather", "Traffic_Level", "Time_of_Day", "Vehicle_Type"]

for i in ohe_cols:
  encoder = OneHotEncoder(sparse_output=True, handle_unknown="ignore")
  encoded = encoder.fit_transform(df[[i]])

  encoded_df = pd.DataFrame(
      encoded.toarray(),
      columns = encoder.get_feature_names_out([i]),
      index = df.index
  )

  df = df.drop(columns = [i])

  df = pd.concat([df, encoded_df], axis=1)

df.head()

In [ ]:
# Task 5: Write your code here:
# Scaling StandardScaler
cols = df.columns.drop("Delivery_Time")
scaler = StandardScaler()
df[cols] = pd.DataFrame(scaler.fit_transform(df[cols]), columns=cols)
df.head()

In [ ]:
# Task 6: Write your code here:

In [ ]:
# Task 1: Write your code here:
X = df.drop("Delivery_Time", axis=1).astype(float)
y = df['Delivery_Time'].astype(float)

In [ ]:
# Task 2,3,4,5: Write your code here:
k = 5

kf = KFold(n_splits=k, shuffle=True, random_state=42)

rf = RandomForestRegressor(n_estimators=200)
rf_mae = []
yPred = []
for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{k}")
  print("Training Random Forest...")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  rf.fit(X_train, y_train)

  y_pred = rf.predict(X_test)
  yPred.append(y_pred)

  mae = mean_absolute_error(y_test, y_pred)

  # Store results
  rf_mae.append(mae)

print(f"\nAverage MAE: {np.mean(rf_mae):.4f}")

In [ ]:
# Task 1: Write your code here:
feature_cols = df.columns.drop("Delivery_Time")
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(yPred, bins=10, edgecolor='black')
plt.title('Distribution')
plt.xlabel('Delivery Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task Bonus: Write your code here:
models = {
  "Random Forest Regressor": RandomForestRegressor(n_estimators=200),
  "CatBoost": CatBoostRegressor(verbose=0)
}

n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)


results = {}

for name in models:
  results[name] = {'mae': []}

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  for model_name, model in models.items():
    print(f"Training {model_name}...")

    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)


    mae = mean_absolute_error(y_test, y_pred)

    # Store results
    results[model_name]["mae"].append(mae)

for model_name in results:
  print(f"\n{model_name}:")
  print(f"MAE:  {np.mean(results[model_name]['mae']):.4f}")